In [1]:
import pandas as pd
df = pd.read_parquet(r"C:\Users\timio\Documents\StatLearningLocal\amex-default-prediction-project\timi_notebooks\train_cleaned.parquet")
features = pd.read_parquet(r"C:\Users\timio\Documents\StatLearningLocal\amex-default-prediction-project\timi_notebooks\features_engineered.parquet")
#print(df.head())

In [2]:
# D_87: 99.9% missing AND constant where present — zero information
features = features.drop(columns=[c for c in features.columns if c.startswith('D_87')])

# Any column that ended up entirely constant post-aggregation
nunique = features.nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
print("Dropping constant columns:", constant_cols)
features = features.drop(columns=constant_cols)

Dropping constant columns: []


In [3]:
missing_pct = df.isna().mean().sort_values(ascending=False) * 100

# Show all columns with any meaningful missingness
print("Columns with >10% missing:")
print(missing_pct[missing_pct > 10])

print(f"\nTotal columns with >10% missing: {(missing_pct > 10).sum()}")
print(f"Total columns with >50% missing: {(missing_pct > 50).sum()}")
print(f"Total columns with any missing:  {(missing_pct > 0).sum()}")
print(f"Total columns with zero missing: {(missing_pct == 0).sum()}")

Columns with >10% missing:
D_87     99.925901
D_88     99.889301
D_108    99.469306
D_111    99.415507
D_110    99.415507
B_39     99.370408
D_73     98.954313
B_42     98.771115
D_134    96.392043
D_138    96.392043
D_135    96.392043
D_136    96.392043
D_137    96.392043
R_9      94.181570
B_29     93.180582
D_106    90.102419
D_132    90.079819
D_49     90.027120
D_76     88.821134
R_26     88.799834
D_66     88.589037
D_42     85.418275
D_142    82.623409
D_53     73.635916
D_82     73.598517
D_50     57.225413
B_17     56.269125
D_105    54.418247
D_56     54.408047
S_9      53.332160
D_77     45.856050
D_43     29.899141
S_27     25.291796
D_46     22.041036
S_3      18.364480
S_7      18.364480
D_62     13.659136
D_48     13.048343
D_61     10.817570
dtype: float64

Total columns with >10% missing: 39
Total columns with >50% missing: 30
Total columns with any missing:  120
Total columns with zero missing: 70


In [4]:
#updated missingness
high_missing = [
    'D_87', 'D_88', 'D_108', 'D_111', 'D_110', 'B_39',
    'D_73', 'B_42', 'D_134', 'D_138', 'D_135', 'D_136',
    'D_137', 'R_9', 'B_29', 'D_106', 'D_132', 'D_49',
    'D_76', 'R_26', 'D_66',
    'D_42', 'D_142', 'D_53', 'D_82', 'D_50',
    'B_17', 'D_105', 'D_56', 'S_9',
    # moderate missingness (>20%)
    'D_77', 'D_43', 'S_27', 'D_46'
]

for c in high_missing:
    last_col = f'{c}_last'
    if last_col in features.columns:
        features[f'{c}_was_missing'] = features[last_col].isna().astype(int)

# Confirm how many indicator columns were added
added = [f'{c}_was_missing' for c in high_missing if f'{c}_last' in features.columns]
print(f"Added {len(added)} missing indicator columns")
print(features.shape)

Added 33 missing indicator columns
(83121, 1107)


In [5]:
import numpy as np
num_cols = df.select_dtypes(include=np.number).columns.tolist()

In [6]:
bimodal_candidates = []

for c in num_cols:
    s = df[c].dropna()
    if len(s) == 0:
        continue
    
    # The bimodal signature we found: median near 0, 99th percentile near 1,
    # and almost nothing in between (very few values in the 0.1-0.9 range)
    median = s.median()
    p99 = s.quantile(0.99)
    pct_middle = ((s > 0.1) & (s < 0.9)).mean()  # fraction of values in the "gap"
    
    if median < 0.05 and p99 > 0.9 and pct_middle < 0.05:
        bimodal_candidates.append({
            'col': c,
            'median': median,
            'p99': p99,
            'pct_in_gap': pct_middle * 100,  # % of values between 0.1 and 0.9
            'pct_near_zero': (s <= 0.1).mean() * 100,
            'pct_near_one': (s >= 0.9).mean() * 100,
        })

bimodal_df = pd.DataFrame(bimodal_candidates).sort_values('pct_in_gap')
print(f"Found {len(bimodal_df)} bimodal candidates:")
print(bimodal_df.to_string())

Found 50 bimodal candidates:
      col    median       p99  pct_in_gap  pct_near_zero  pct_near_one
0     R_2  0.005227  1.007690    0.000000      95.623853      4.376147
27   B_32  0.005121  1.005533    0.000000      97.759327      2.240673
28   S_20  0.005062  1.002074    0.000000      98.740715      1.259285
29   R_20  0.005078  1.006393    0.000000      98.449498      1.550502
30   R_21  0.005081  1.004066    0.000000      98.307520      1.692480
31   D_92  0.005442  1.009214    0.000000      91.909297      8.090703
32   D_93  0.005054  1.000019    0.000000      98.997212      1.002788
33   D_94  0.005084  1.003893    0.000000      98.350720      1.649280
34   R_24  0.005066  1.003070    0.000000      98.564717      1.435283
35   D_96  0.005158  1.006790    0.000000      96.921437      3.078563
36  D_103  0.009331  1.009781    0.000000      53.549538     46.450462
37  D_104  0.009338  1.005307    0.000000      53.549538     46.450462
38  D_108  0.005326  1.009893    0.000000      9

In [7]:
#Binary Flag Code for bimodal columns

# Group 1 & 2: clean bimodal (pct_in_gap == 0) — reliable binary flag
clean_bimodal = bimodal_df[bimodal_df['pct_in_gap'] == 0]['col'].tolist()

# Group 3: fuzzy bimodal (pct_in_gap > 0) — add flag but less critical
fuzzy_bimodal = bimodal_df[bimodal_df['pct_in_gap'] > 0]['col'].tolist()

all_bimodal = clean_bimodal + fuzzy_bimodal

for c in all_bimodal:
    last_col = f'{c}_last'
    if last_col in features.columns:
        features[f'{c}_is_high'] = (features[last_col] > 0.5).astype(int)

print(f"Added binary flags for {len([c for c in all_bimodal if f'{c}_last' in features.columns])} bimodal columns")
print(features.shape)

# Justification: A linear model won't naturally learn the threshold effect in a bimodal 
#column — the relationship between the raw value and target isn't linear, it's 
#step-function-shaped. The explicit binary flag makes that threshold directly available 
#to the model. Tree-based models will find this themselves anyway, so the flag is a 
#helpful nudge for linear models but doesn't hurt trees either.

Added binary flags for 50 bimodal columns
(83121, 1157)


In [8]:
cat_cols = ['B_30', 'B_38', 'D_114', 'D_116', 'D_117', 'D_120', 'D_126', 'D_63', 'D_64', 'D_66', 'D_68']
present = [c for c in cat_cols if c in df.columns]

for c in present:
    print(f"\n{c}: {sorted(df[c].dropna().unique())}")


B_30: ['0', '1', '2']

B_38: ['1', '2', '3', '4', '5', '6', '7']

D_114: ['0', '1']

D_116: ['0', '1']

D_117: ['-1', '1', '2', '3', '4', '5', '6']

D_120: ['0', '1']

D_126: ['-1', '0', '1']

D_63: ['CL', 'CO', 'CR', 'XL', 'XM', 'XZ']

D_64: ['-1', 'O', 'R', 'U']

D_66: ['0', '1']

D_68: ['0', '1', '2', '3', '4', '5', '6']


In [9]:
#Categorical Encoder
# Explicitly target only the _last and _mode categorical aggregation columns
cat_agg_cols = [f'{cat}_{suffix}' 
                for cat in present 
                for suffix in ['last', 'mode']
                if f'{cat}_{suffix}' in features.columns]

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for c in cat_agg_cols:
    features[c] = features[c].fillna('MISSING')
    features[c] = le.fit_transform(features[c].astype(str))

In [10]:
# Impute remaining missing values

from sklearn.impute import SimpleImputer
#import numpy as np

# Separate numeric and categorical post-encoding
num_feature_cols = features.select_dtypes(include=np.number).columns.drop(['target'], errors='ignore')

# Median imputation: robust to outliers (important given B_10, B_6 scale we found)
# Note: LightGBM can handle NaN natively — if using LGBM, you can skip this entirely
imputer = SimpleImputer(strategy='median')
features[num_feature_cols] = imputer.fit_transform(features[num_feature_cols])

In [11]:
#Scaler already exists in library

# from sklearn.preprocessing import StandardScaler

# # Identify binary columns to exclude from scaling
# binary_cols = (
#     [f'{c}_was_missing' for c in high_missing if f'{c}_was_missing' in features.columns] +
#     [f'{c}_is_high' for c in all_bimodal if f'{c}_is_high' in features.columns]
# )

# # Only scale genuine continuous columns
# scale_cols = [c for c in features.select_dtypes(include=np.number).columns
#               if c not in binary_cols
#               and c != 'target']

# scaler = StandardScaler()
# features[scale_cols] = scaler.fit_transform(features[scale_cols])

In [12]:
print('customer_ID' in features.columns)
print('target' in features.columns)
print(features.columns.tolist()[:10])  # first 10 column names

True
False
['customer_ID', 'P_2_last', 'P_2_mean', 'P_2_std', 'P_2_min', 'P_2_max', 'D_39_last', 'D_39_mean', 'D_39_std', 'D_39_min']


In [13]:
labels = pd.read_parquet(r"C:\Users\timio\Documents\StatLearningLocal\amex-default-prediction-project\data\interim\train_1m_labels.parquet")

In [14]:
features = features.merge(labels[['customer_ID', 'target']], on='customer_ID', how='left')
print('target' in features.columns)  # should now be True
print(features.shape)

True
(83121, 1158)


In [15]:
# This block does two things: sets up the ingredients needed for model training, and 
#defines a reusable function that runs cross-validation for any model you hand it.

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# --- Setup ---
X = features.drop(columns=['customer_ID', 'target'])
y = features['target']

# StratifiedKFold preserves class balance in each fold — critical for imbalanced data
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Storage for results
results = {}

def run_cv(model, model_name, X, y, skf):
    oof_preds = np.zeros(len(y))  # out-of-fold predictions
    fold_aucs = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        
        val_preds = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_preds
        
        fold_auc = roc_auc_score(y_val, val_preds)
        fold_aucs.append(fold_auc)
        print(f"  Fold {fold+1} AUC: {fold_auc:.4f}")
    
    overall_auc = roc_auc_score(y, oof_preds)
    print(f"  {model_name} Overall OOF AUC: {overall_auc:.4f} | Std: {np.std(fold_aucs):.4f}\n")
    
    return oof_preds, overall_auc

In [16]:
# --- Model 1: Logistic Regression (baseline) ---
print("=== Logistic Regression ===")
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_preds, lr_auc = run_cv(lr, 'Logistic Regression', X, y, skf)
results['Logistic Regression'] = lr_auc

=== Logistic Regression ===
  Fold 1 AUC: 0.9568
  Fold 2 AUC: 0.9567
  Fold 3 AUC: 0.9540
  Fold 4 AUC: 0.9539
  Fold 5 AUC: 0.9557
  Logistic Regression Overall OOF AUC: 0.9554 | Std: 0.0012



In [17]:
# --- Model 2: Random Forest ---
print("=== Random Forest ===")
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_preds, rf_auc = run_cv(rf, 'Random Forest', X, y, skf)
results['Random Forest'] = rf_auc

=== Random Forest ===
  Fold 1 AUC: 0.9520
  Fold 2 AUC: 0.9505
  Fold 3 AUC: 0.9486
  Fold 4 AUC: 0.9476
  Fold 5 AUC: 0.9497
  Random Forest Overall OOF AUC: 0.9497 | Std: 0.0015



In [18]:
# --- Model 3: LightGBM ---
print("=== LightGBM ===")

lgb_oof = np.zeros(len(y))
fold_aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data   = lgb.Dataset(X_val,   label=y_val)
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'learning_rate': 0.05,
        'num_leaves': 64,
        'min_child_samples': 50,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 1,
        'scale_pos_weight': (y == 0).sum() / (y == 1).sum(),  # handles class imbalance
        'verbosity': -1,
        'random_state': 42
    }
    
    model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )
    
    val_preds = model.predict(X_val)
    lgb_oof[val_idx] = val_preds
    
    fold_auc = roc_auc_score(y_val, val_preds)
    fold_aucs.append(fold_auc)
    print(f"  Fold {fold+1} AUC: {fold_auc:.4f} | Best round: {model.best_iteration}")

lgb_auc = roc_auc_score(y, lgb_oof)
print(f"  LightGBM Overall OOF AUC: {lgb_auc:.4f} | Std: {np.std(fold_aucs):.4f}\n")
results['LightGBM'] = lgb_auc

=== LightGBM ===
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.958464
[200]	valid_0's auc: 0.959671
[300]	valid_0's auc: 0.959807
Early stopping, best iteration is:
[256]	valid_0's auc: 0.959912
  Fold 1 AUC: 0.9599 | Best round: 256
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.956997
[200]	valid_0's auc: 0.958269
Early stopping, best iteration is:
[225]	valid_0's auc: 0.958305
  Fold 2 AUC: 0.9583 | Best round: 225
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.955577
[200]	valid_0's auc: 0.956398
Early stopping, best iteration is:
[181]	valid_0's auc: 0.956501
  Fold 3 AUC: 0.9565 | Best round: 181
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.954327
[200]	valid_0's auc: 0.955711
[300]	valid_0's auc: 0.955806
Early stopping, best iteration is:
[271]	valid_0's auc: 0.955933
  Fold 4 AUC: 0.9559 | Best round: 271
Training until validati

In [19]:
# --- Model 4: XGBoost ---
print("=== XGBoost ===")

xgb_oof = np.zeros(len(y))
fold_aucs = []
scale_pos = (y == 0).sum() / (y == 1).sum()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos,
        eval_metric='auc',
        early_stopping_rounds=50,
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=100
    )
    
    val_preds = model.predict_proba(X_val)[:, 1]
    xgb_oof[val_idx] = val_preds
    
    fold_auc = roc_auc_score(y_val, val_preds)
    fold_aucs.append(fold_auc)
    print(f"  Fold {fold+1} AUC: {fold_auc:.4f}")

xgb_auc = roc_auc_score(y, xgb_oof)
print(f"  XGBoost Overall OOF AUC: {xgb_auc:.4f} | Std: {np.std(fold_aucs):.4f}\n")
results['XGBoost'] = xgb_auc

=== XGBoost ===
[0]	validation_0-auc:0.93328
[100]	validation_0-auc:0.95711
[200]	validation_0-auc:0.95914
[300]	validation_0-auc:0.95939
[400]	validation_0-auc:0.95954
[432]	validation_0-auc:0.95958
  Fold 1 AUC: 0.9596
[0]	validation_0-auc:0.93276
[100]	validation_0-auc:0.95623
[200]	validation_0-auc:0.95823
[300]	validation_0-auc:0.95861
[400]	validation_0-auc:0.95878
[500]	validation_0-auc:0.95887
[536]	validation_0-auc:0.95888
  Fold 2 AUC: 0.9589
[0]	validation_0-auc:0.93206
[100]	validation_0-auc:0.95454
[200]	validation_0-auc:0.95642
[300]	validation_0-auc:0.95678
[353]	validation_0-auc:0.95668
  Fold 3 AUC: 0.9568
[0]	validation_0-auc:0.92899
[100]	validation_0-auc:0.95344
[200]	validation_0-auc:0.95545
[300]	validation_0-auc:0.95568
[400]	validation_0-auc:0.95574
[441]	validation_0-auc:0.95574
  Fold 4 AUC: 0.9558
[0]	validation_0-auc:0.93123
[100]	validation_0-auc:0.95490
[200]	validation_0-auc:0.95665
[300]	validation_0-auc:0.95692
[400]	validation_0-auc:0.95713
[463]	valid

In [20]:
# from sklearn.naive_bayes import GaussianNB, BernoulliNB
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import MinMaxScaler
# import numpy as np

# # Identify binary columns (the _was_missing and _is_high flags we created)
# binary_cols = [c for c in X.columns if c.endswith('_was_missing') or c.endswith('_is_high')]
# continuous_cols = [c for c in X.columns if c not in binary_cols]

# # Option A: GaussianNB — assumes continuous features follow a normal distribution
# # Not ideal here (we know many columns are skewed/bimodal) but simplest to run
# print("=== Naive Bayes (Gaussian) ===")
# gnb = GaussianNB()
# gnb_preds, gnb_auc = run_cv(gnb, 'Gaussian Naive Bayes', X, y, skf)
# results['Gaussian NB'] = gnb_auc

In [21]:
# # Option B: Separate treatment for binary vs continuous features
# # This is more correct given our data structure
# from sklearn.naive_bayes import GaussianNB, BernoulliNB
# from scipy.sparse import hstack
# import numpy as np

# gnb_oof = np.zeros(len(y))
# bnb_oof = np.zeros(len(y))
# combined_oof = np.zeros(len(y))
# fold_aucs = []

# for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
#     # Gaussian NB on continuous features
#     gnb = GaussianNB()
#     gnb.fit(X_train[continuous_cols].fillna(0), y_train)
#     cont_preds = gnb.predict_proba(X_val[continuous_cols].fillna(0))[:, 1]
    
#     # Bernoulli NB on binary features
#     bnb = BernoulliNB()
#     bnb.fit(X_train[binary_cols].fillna(0), y_train)
#     bin_preds = bnb.predict_proba(X_val[binary_cols].fillna(0))[:, 1]
    
#     # Average the two sets of predictions
#     val_preds = (cont_preds + bin_preds) / 2
#     combined_oof[val_idx] = val_preds
    
#     fold_auc = roc_auc_score(y_val, val_preds)
#     fold_aucs.append(fold_auc)
#     print(f"  Fold {fold+1} AUC: {fold_auc:.4f}")

# nb_auc = roc_auc_score(y, combined_oof)
# print(f"  Naive Bayes (Combined) OOF AUC: {nb_auc:.4f} | Std: {np.std(fold_aucs):.4f}\n")
# results['Naive Bayes'] = nb_auc

In [22]:
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

binary_cols = [c for c in X.columns if c.endswith('_was_missing') or c.endswith('_is_high')]
continuous_cols = [c for c in X.columns if c not in binary_cols]

nb_oof = np.zeros(len(y))
fold_roc_aucs = []
fold_pr_aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Gaussian NB on continuous features
    gnb = GaussianNB()
    gnb.fit(X_train[continuous_cols].fillna(0), y_train)
    cont_preds = gnb.predict_proba(X_val[continuous_cols].fillna(0))[:, 1]
    
    # Bernoulli NB on binary features
    bnb = BernoulliNB()
    bnb.fit(X_train[binary_cols].fillna(0), y_train)
    bin_preds = bnb.predict_proba(X_val[binary_cols].fillna(0))[:, 1]
    
    val_preds = (cont_preds + bin_preds) / 2
    nb_oof[val_idx] = val_preds
    
    fold_roc = roc_auc_score(y_val, val_preds)
    fold_pr  = average_precision_score(y_val, val_preds)
    fold_roc_aucs.append(fold_roc)
    fold_pr_aucs.append(fold_pr)
    
    print(f"  Fold {fold+1} ROC AUC: {fold_roc:.4f} | PR AUC: {fold_pr:.4f}")

overall_roc = roc_auc_score(y, nb_oof)
overall_pr  = average_precision_score(y, nb_oof)
print(f"\n  Naive Bayes OOF ROC AUC: {overall_roc:.4f} | Std: {np.std(fold_roc_aucs):.4f}")
print(f"  Naive Bayes OOF PR AUC:  {overall_pr:.4f}  | Std: {np.std(fold_pr_aucs):.4f}")

results['Naive Bayes'] = {'roc_auc': overall_roc, 'pr_auc': overall_pr}

  Fold 1 ROC AUC: 0.9162 | PR AUC: 0.8050
  Fold 2 ROC AUC: 0.9162 | PR AUC: 0.8091
  Fold 3 ROC AUC: 0.9079 | PR AUC: 0.7917
  Fold 4 ROC AUC: 0.9132 | PR AUC: 0.7971
  Fold 5 ROC AUC: 0.9137 | PR AUC: 0.8003

  Naive Bayes OOF ROC AUC: 0.9134 | Std: 0.0030
  Naive Bayes OOF PR AUC:  0.8005  | Std: 0.0060


In [23]:
#!pip install torch

In [24]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

X_nn = X.fillna(0)
scaler_nn = StandardScaler()
X_nn_scaled = scaler_nn.fit_transform(X_nn)

class TabularNet(nn.Module):
    def __init__(self, input_dim):
        super(TabularNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x).squeeze()

In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

nn_oof = np.zeros(len(y))
fold_roc_aucs = []
fold_pr_aucs  = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_nn_scaled, y)):
    print(f"\n  Fold {fold+1}")
    
    X_train_t = torch.FloatTensor(X_nn_scaled[train_idx]).to(device)
    y_train_t = torch.FloatTensor(y.iloc[train_idx].values).to(device)
    X_val_t   = torch.FloatTensor(X_nn_scaled[val_idx]).to(device)
    y_val_np  = y.iloc[val_idx].values  # keep as numpy for sklearn metrics
    
    train_ds = TensorDataset(X_train_t, y_train_t)
    train_dl = DataLoader(train_ds, batch_size=1024, shuffle=True)
    
    model_nn = TabularNet(input_dim=X_nn_scaled.shape[1]).to(device)
    
    pos_weight = torch.tensor([(y == 0).sum() / (y == 1).sum()]).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model_nn.parameters(), lr=1e-3, weight_decay=1e-5)
    
    # ReduceLROnPlateau now monitors PR AUC instead of ROC AUC
    # PR AUC is more sensitive to improvements on the minority class
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )
    
    best_pr_auc   = 0
    best_roc_auc  = 0
    patience_counter = 0
    best_weights  = None
    
    for epoch in range(50):
        # Training pass
        model_nn.train()
        for X_batch, y_batch in train_dl:
            optimizer.zero_grad()
            preds = model_nn(X_batch)
            loss  = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
        
        # Validation pass
        model_nn.eval()
        with torch.no_grad():
            val_preds_np = model_nn(X_val_t).cpu().numpy()
        
        val_roc = roc_auc_score(y_val_np, val_preds_np)
        val_pr  = average_precision_score(y_val_np, val_preds_np)
        
        # Use PR AUC to drive early stopping and LR scheduling
        # (more meaningful than ROC AUC for imbalanced data)
        scheduler.step(val_pr)
        
        if val_pr > best_pr_auc:
            best_pr_auc  = val_pr
            best_roc_auc = val_roc
            best_weights = model_nn.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= 10:
            print(f"    Early stop at epoch {epoch+1} | "
                  f"Best ROC AUC: {best_roc_auc:.4f} | Best PR AUC: {best_pr_auc:.4f}")
            break
    
    # Load best weights and make final predictions
    model_nn.load_state_dict(best_weights)
    model_nn.eval()
    with torch.no_grad():
        final_preds = model_nn(X_val_t).cpu().numpy()
    
    nn_oof[val_idx] = final_preds
    fold_roc_aucs.append(best_roc_auc)
    fold_pr_aucs.append(best_pr_auc)
    print(f"  Fold {fold+1} Best ROC AUC: {best_roc_auc:.4f} | Best PR AUC: {best_pr_auc:.4f}")

overall_roc = roc_auc_score(y, nn_oof)
overall_pr  = average_precision_score(y, nn_oof)
print(f"\n  Neural Network OOF ROC AUC: {overall_roc:.4f} | Std: {np.std(fold_roc_aucs):.4f}")
print(f"  Neural Network OOF PR AUC:  {overall_pr:.4f}  | Std: {np.std(fold_pr_aucs):.4f}")

results['Neural Network'] = {'roc_auc': overall_roc, 'pr_auc': overall_pr}

Using device: cpu

  Fold 1
    Early stop at epoch 12 | Best ROC AUC: 0.9481 | Best PR AUC: 0.8800
  Fold 1 Best ROC AUC: 0.9481 | Best PR AUC: 0.8800

  Fold 2
    Early stop at epoch 16 | Best ROC AUC: 0.9442 | Best PR AUC: 0.8790
  Fold 2 Best ROC AUC: 0.9442 | Best PR AUC: 0.8790

  Fold 3
    Early stop at epoch 17 | Best ROC AUC: 0.9428 | Best PR AUC: 0.8758
  Fold 3 Best ROC AUC: 0.9428 | Best PR AUC: 0.8758

  Fold 4
    Early stop at epoch 11 | Best ROC AUC: 0.9434 | Best PR AUC: 0.8717
  Fold 4 Best ROC AUC: 0.9434 | Best PR AUC: 0.8717

  Fold 5
    Early stop at epoch 11 | Best ROC AUC: 0.9484 | Best PR AUC: 0.8782
  Fold 5 Best ROC AUC: 0.9484 | Best PR AUC: 0.8782

  Neural Network OOF ROC AUC: 0.9283 | Std: 0.0024
  Neural Network OOF PR AUC:  0.8634  | Std: 0.0030


In [26]:
print("Number of features:", features.shape[1])
print("Total columns in features:", features.shape)

Number of features: 1158
Total columns in features: (83121, 1158)


In [27]:
from sklearn.tree import DecisionTreeClassifier

print("=== Decision Tree ===")
dt = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=42,
    max_depth=10,        # without a depth limit, trees overfit badly on 950 features
    min_samples_leaf=50  # each leaf must have at least 50 customers
)
dt_preds, dt_auc = run_cv(dt, 'Decision Tree', X, y, skf)
results['Decision Tree'] = dt_auc

=== Decision Tree ===
  Fold 1 AUC: 0.9332
  Fold 2 AUC: 0.9296
  Fold 3 AUC: 0.9268
  Fold 4 AUC: 0.9255
  Fold 5 AUC: 0.9277
  Decision Tree Overall OOF AUC: 0.9286 | Std: 0.0027



In [29]:
!pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------  2.1/2.1 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 10.1 MB/s  0:00:00

   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ----- ---------------------------------- 1/7 [Mako]
   ----- ---------------------------------- 1/7 [Mako]
   ----- ---------------------------------- 1/7 [Mako]
   ----- ---------------------------------- 1/7 [Mako]
   ----- ---------------------------------- 1/7 [Mako]
   ----------- ---------------------------- 2/7 [greenlet]
   ----------- ---------------------------- 2/7 [greenlet]
   ----------- ---------------------------- 2/7 [greenlet]
   ----------------- ---------------------- 3/7 [colorlog]
   ---------------------- ----------------- 4/7 [sqlalchemy]
   ---------------------- ----------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 150),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': 1,
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),  # L1 regularization
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True), # L2 regularization
        'scale_pos_weight': (y == 0).sum() / (y == 1).sum(),
        'random_state': 42
    }
    
    fold_aucs = []
    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.train(
            params,
            lgb.Dataset(X_train, label=y_train),
            num_boost_round=1000,
            valid_sets=[lgb.Dataset(X_val, label=y_val)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(-1)]
        )
        fold_aucs.append(roc_auc_score(y_val, model.predict(X_val)))
    
    return np.mean(fold_aucs)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Best ROC AUC:", study.best_value)
print("Best params:", study.best_params)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[886]	valid_0's auc: 0.961031
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[999]	valid_0's auc: 0.959868
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[992]	valid_0's auc: 0.958007
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's auc: 0.957022
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[950]	valid_0's auc: 0.958822
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[993]	valid_0's auc: 0.960945
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[982]	valid_0's auc: 0.959958
Training until validation scores don't improve for 5

In [ ]:
# # Neural Networks Tuning:

# # Two changes from the original:
# # 1. Lower learning rate: 1e-3 → 3e-4
# # 2. More patience: 10 → 20 epochs before early stopping

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Using device: {device}")

# nn_oof = np.zeros(len(y))
# fold_roc_aucs = []
# fold_pr_aucs  = []

# for fold, (train_idx, val_idx) in enumerate(skf.split(X_nn_scaled, y)):
#     print(f"\n  Fold {fold+1}")
    
#     X_train_t = torch.FloatTensor(X_nn_scaled[train_idx]).to(device)
#     y_train_t = torch.FloatTensor(y.iloc[train_idx].values).to(device)
#     X_val_t   = torch.FloatTensor(X_nn_scaled[val_idx]).to(device)
#     y_val_np  = y.iloc[val_idx].values  # keep as numpy for sklearn metrics
    
#     train_ds = TensorDataset(X_train_t, y_train_t)
#     train_dl = DataLoader(train_ds, batch_size=1024, shuffle=True)
    
#     model_nn = TabularNet(input_dim=X_nn_scaled.shape[1]).to(device)
    
#     pos_weight = torch.tensor([(y == 0).sum() / (y == 1).sum()]).to(device)
#     criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
#     optimizer  = torch.optim.Adam(model_nn.parameters(), lr=1e-4, weight_decay=1e-5)
    
#     # ReduceLROnPlateau now monitors PR AUC instead of ROC AUC
#     # PR AUC is more sensitive to improvements on the minority class
#     scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer, mode='max', factor=0.5, patience=5  # reduce LR after 5 epochs of no improvement
#     )
    
#     best_pr_auc   = 0
#     best_roc_auc  = 0
#     patience_counter = 0
#     best_weights  = None
    
#     for epoch in range(50):
#         # Training pass
#         model_nn.train()
#         for X_batch, y_batch in train_dl:
#             optimizer.zero_grad()
#             preds = model_nn(X_batch)
#             loss  = criterion(preds, y_batch)
#             loss.backward()
#             optimizer.step()
        
#         # Validation pass
#         model_nn.eval()
#         with torch.no_grad():
#             val_preds_np = model_nn(X_val_t).cpu().numpy()
        
#         val_roc = roc_auc_score(y_val_np, val_preds_np)
#         val_pr  = average_precision_score(y_val_np, val_preds_np)
        
#         # Use PR AUC to drive early stopping and LR scheduling
#         # (more meaningful than ROC AUC for imbalanced data)
#         scheduler.step(val_pr)
        
#         if val_pr > best_pr_auc:
#             best_pr_auc  = val_pr
#             best_roc_auc = val_roc
#             best_weights = model_nn.state_dict().copy()
#             patience_counter = 0
#         else:
#             patience_counter += 1
        
#         if patience_counter >= 20: # was 10, now 20
#             print(f"    Early stop at epoch {epoch+1} | "
#                   f"Best ROC AUC: {best_roc_auc:.4f} | Best PR AUC: {best_pr_auc:.4f}")
#             break
    
#     # Load best weights and make final predictions
#     model_nn.load_state_dict(best_weights)
#     model_nn.eval()
#     with torch.no_grad():
#         final_preds = model_nn(X_val_t).cpu().numpy()
    
#     nn_oof[val_idx] = final_preds
#     fold_roc_aucs.append(best_roc_auc)
#     fold_pr_aucs.append(best_pr_auc)
#     print(f"  Fold {fold+1} Best ROC AUC: {best_roc_auc:.4f} | Best PR AUC: {best_pr_auc:.4f}")

# overall_roc = roc_auc_score(y, nn_oof)
# overall_pr  = average_precision_score(y, nn_oof)
# print(f"\n  Neural Network OOF ROC AUC: {overall_roc:.4f} | Std: {np.std(fold_roc_aucs):.4f}")
# print(f"  Neural Network OOF PR AUC:  {overall_pr:.4f}  | Std: {np.std(fold_pr_aucs):.4f}")

# results['Neural Network'] = {'roc_auc': overall_roc, 'pr_auc': overall_pr}

In [ ]:
from sklearn.linear_model import LogisticRegression as LR

stack_train = pd.DataFrame({
    'lgb': lgb_oof,
    'lr':  lr_preds,
    'nn':  nn_oof,
})

stack_oof = np.zeros(len(y))
fold_roc_aucs = []

meta = LR()
for fold, (train_idx, val_idx) in enumerate(skf.split(stack_train, y)):
    meta.fit(stack_train.iloc[train_idx], y.iloc[train_idx])
    val_preds = meta.predict_proba(stack_train.iloc[val_idx])[:, 1]
    stack_oof[val_idx] = val_preds
    
    fold_roc = roc_auc_score(y.iloc[val_idx], val_preds)
    fold_roc_aucs.append(fold_roc)
    print(f"  Fold {fold+1} ROC AUC: {fold_roc:.4f}")

stack_roc = roc_auc_score(y, stack_oof)
print(f"\nStacked Model OOF ROC AUC: {stack_roc:.4f} | Std: {np.std(fold_roc_aucs):.4f}")
results['Stacked'] = stack_roc

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

# --- 1. Summary comparison table ---
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'LightGBM', 
              'XGBoost', 'Naive Bayes', 'Neural Network', 
              'Decision Tree', 'Stacked'],
    'ROC AUC': [lr_auc, rf_auc, lgb_auc, xgb_auc, 
                nb_auc, nn_auc, dt_auc, stack_roc],
    'Std': [0.0012, 0.0015, 0.0014, 0.0014, 
            0.0030, 0.0024, 0.0027, None]
}).sort_values('ROC AUC', ascending=False).reset_index(drop=True)

comparison['Rank'] = comparison.index + 1
print(comparison.to_string(index=False))

In [ ]:
# --- 2. ROC curves for all models on one plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

model_preds = {
    'Logistic Regression': lr_preds,
    'Random Forest':       rf_preds,
    'LightGBM':            lgb_oof,
    'XGBoost':             xgb_oof,
    'Naive Bayes':         nb_oof,
    'Neural Network':      nn_oof,
    'Decision Tree':       dt_preds,
    'Stacked':             stack_oof,
}

# ROC curves
for name, preds in model_preds.items():
    fpr, tpr, _ = roc_curve(y, preds)
    auc = roc_auc_score(y, preds)
    axes[0].plot(fpr, tpr, label=f'{name} ({auc:.4f})')

axes[0].plot([0,1], [0,1], 'k--', label='Random (0.5000)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — All Models')
axes[0].legend(loc='lower right', fontsize=8)
axes[0].grid(True)

# PR curves
for name, preds in model_preds.items():
    precision, recall, _ = precision_recall_curve(y, preds)
    pr_auc = average_precision_score(y, preds)
    axes[1].plot(recall, precision, label=f'{name} ({pr_auc:.4f})')

baseline = y.mean()  # default rate = random PR baseline
axes[1].axhline(y=baseline, color='k', linestyle='--', 
                label=f'Random ({baseline:.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curves — All Models')
axes[1].legend(loc='upper right', fontsize=8)
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- 3. Is the best model meaningfully better? ---
# Check if differences between top models are larger than their std
# Rule of thumb: difference > 2x std = meaningful gap

print("Gap analysis (vs. best model):")
best_auc = comparison['ROC AUC'].iloc[0]
for _, row in comparison.iterrows():
    gap = best_auc - row['ROC AUC']
    std = row['Std'] if row['Std'] else 0
    meaningful = "✓ meaningful" if gap > 2 * std else "~ within noise" if gap > 0 else "← BEST"
    print(f"  {row['Model']:<25} gap: {gap:.4f}  {meaningful}")

In [ ]:
# --- 4. Confusion matrix at optimal threshold for best model ---
from sklearn.metrics import precision_recall_curve

# Find threshold that maximizes F1
best_preds = lgb_oof  # replace with whichever model wins
precisions, recalls, thresholds = precision_recall_curve(y, best_preds)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Optimal threshold: {best_threshold:.3f}")

binary_preds = (best_preds >= best_threshold).astype(int)
cm = confusion_matrix(y, binary_preds)
ConfusionMatrixDisplay(cm, display_labels=['No Default', 'Default']).plot()
plt.title(f'Best Model Confusion Matrix (threshold={best_threshold:.3f})')
plt.show()

from sklearn.metrics import classification_report
print(classification_report(y, binary_preds, target_names=['No Default', 'Default']))

In [ ]:
# --- Approach 1: LightGBM built-in importance ---
# Refit on full data to get stable importances
best_lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'learning_rate': 0.05,
    'num_leaves': 64,
    'scale_pos_weight': (y == 0).sum() / (y == 1).sum(),
    'random_state': 42
}

final_lgb = lgb.train(
    best_lgb_params,
    lgb.Dataset(X, label=y),
    num_boost_round=250  # approximate average of best rounds across folds
)

importance_df = pd.DataFrame({
    'feature':    X.columns,
    'importance_gain':  final_lgb.feature_importance(importance_type='gain'),
    'importance_split': final_lgb.feature_importance(importance_type='split'),
}).sort_values('importance_gain', ascending=False).reset_index(drop=True)

importance_df['rank'] = importance_df.index + 1
print("Top 30 most important features:")
print(importance_df.head(30).to_string(index=False))

In [ ]:
# Plot top 30
plt.figure(figsize=(10, 10))
top30 = importance_df.head(30)
plt.barh(top30['feature'][::-1], top30['importance_gain'][::-1])
plt.xlabel('Importance (Gain)')
plt.title('Top 30 Features by Importance (LightGBM Gain)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Rank all customers by predicted default probability ---
customer_scores = pd.DataFrame({
    'customer_ID':    features['customer_ID'],
    'default_prob':   lgb_oof,  # use best model's OOF predictions
    'actual_default': y.values
}).sort_values('default_prob', ascending=False).reset_index(drop=True)

customer_scores['risk_rank'] = customer_scores.index + 1  # 1 = highest risk

# Add risk tier labels
customer_scores['risk_tier'] = pd.cut(
    customer_scores['default_prob'],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)

print("Customer risk ranking (top 20 highest risk):")
print(customer_scores.head(20).to_string(index=False))
print()
print("Risk tier distribution:")
print(customer_scores['risk_tier'].value_counts().sort_index())

In [ ]:
# --- How well does our ranking capture actual defaults? ---
# Lift curve: among the top X% of predicted high-risk customers,
# what fraction are actual defaulters?

total_defaults = y.sum()
lift_results = []

for pct in [0.05, 0.10, 0.20, 0.30, 0.50]:
    n = int(len(customer_scores) * pct)
    top_n = customer_scores.head(n)
    defaults_captured = top_n['actual_default'].sum()
    lift = (defaults_captured / n) / (total_defaults / len(y))
    lift_results.append({
        'Top %': f'Top {int(pct*100)}%',
        'Customers': n,
        'Defaults captured': int(defaults_captured),
        '% of all defaults': f"{defaults_captured/total_defaults*100:.1f}%",
        'Lift': f"{lift:.2f}x"
    })

print("Lift analysis:")
print(pd.DataFrame(lift_results).to_string(index=False))

In [ ]:
# # Save the ranked customer list
# customer_scores.to_csv(r"path\to\your\folder\customer_risk_rankings.csv", index=False)
# print("Saved customer risk rankings.")